<a href="https://colab.research.google.com/github/giTan7/Adaptive-Stress-Monitoring-on-Wearable-Devices/blob/main/Preproc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stress Dataset Preprocessing Pipeline

This notebook contains the preprocessing pipeline used for multimodal stress detection experiments on the InHouse and WESAD datasets.

The preprocessing workflow includes:

* raw physiological signal loading
* timestamp synchronization and resampling
* signal cleaning and interpolation
* statistical feature extraction
* contextual feature generation
* stress label construction
* categorical feature encoding
* final ML-ready dataset generation

The processed output consists of synchronized multimodal feature tables containing physiological, behavioral, temporal, and stress-related information for downstream model training and inference.

Before running the notebook:

* update dataset input/output paths
* verify dataset folder structure
* ensure required Python packages are installed

Output:

* subject-wise processed CSV files ready for machine learning experiments.

> stress_daily has only 5,982 labels out of 68,460 rows

> Tensor Shape (Inhouse)
> X ∈ R^(Batch × Time × Features)
> (32 × 15 × 8)

* 32 = batch size
* 15 = timesteps
* 8 = features/channels


1. Imports and Configuration

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [ ]:
# =========================
# CONFIGURATION
# =========================

BASE_RAW_PATH = r"path\Inhouse RAW"
OUTPUT_PATH = r"path\InHouse\processed"

RESAMPLE_FREQ = '30S'
LABEL_RATIO = 0.1

os.makedirs(OUTPUT_PATH, exist_ok=True)

2. Load Raw Signals

In [ ]:
# Heart Rate (HR)

def load_hr(subject_path):
    hr_path = os.path.join(subject_path, "Fitbit", "HR")

    if not os.path.exists(hr_path):
        return pd.DataFrame()

    hr_files = [
        f for f in os.listdir(hr_path)
        if f.startswith("heart_rate-") and f.endswith(".json")
    ]

    records = []

    for file in hr_files:
        file_path = os.path.join(hr_path, file)

        try:
            with open(file_path, 'r') as f:
                entries = json.load(f)

            for entry in entries:
                timestamp = datetime.strptime(
                    entry['dateTime'],
                    '%m/%d/%y %H:%M:%S'
                )

                bpm = entry['value']['bpm']

                records.append({
                    'timestamp': timestamp,
                    'hr': bpm
                })

        except Exception:
            continue

    if not records:
        return pd.DataFrame()

    df = pd.DataFrame(records)

    return (
        df
        .drop_duplicates('timestamp')
        .sort_values('timestamp')
        .set_index('timestamp')
    )

In [ ]:
# Electrodermal Activity (EDA)

def load_eda(subject_path):
    eda_path = os.path.join(subject_path, "Fitbit", "EDA")

    if not os.path.exists(eda_path):
        return pd.DataFrame()

    eda_files = [f for f in os.listdir(eda_path) if f.endswith('.csv')]

    if not eda_files:
        return pd.DataFrame()

    file_path = os.path.join(eda_path, eda_files[0])

    df = pd.read_csv(file_path)

    if 'eda_level_real' not in df.columns:
        return pd.DataFrame()

    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df['timestamp'] = (
        df['timestamp']
        .dt.tz_convert('Asia/Kolkata')
        .dt.tz_localize(None)
    )

    df = df[['timestamp', 'eda_level_real']]
    df = df.rename(columns={'eda_level_real': 'eda'})

    return (
        df
        .drop_duplicates('timestamp')
        .sort_values('timestamp')
        .set_index('timestamp')
    )

In [ ]:
# Temperature

def load_temperature(subject_path):
    temp_path = os.path.join(subject_path, "Fitbit", "Temperature")

    if not os.path.exists(temp_path):
        return pd.DataFrame()

    temp_files = [f for f in os.listdir(temp_path) if f.endswith('.csv')]

    dfs = []

    for file in temp_files:
        file_path = os.path.join(temp_path, file)

        try:
            df = pd.read_csv(file_path)

            if 'temperature' in df.columns and 'recorded_time' in df.columns:
                df['timestamp'] = pd.to_datetime(df['recorded_time'])
                dfs.append(df[['timestamp', 'temperature']])

        except Exception:
            continue

    if not dfs:
        return pd.DataFrame()

    df = pd.concat(dfs)

    return (
        df
        .drop_duplicates('timestamp')
        .sort_values('timestamp')
        .set_index('timestamp')
    )

3. Signal Synchronization and Resampling

In [ ]:
def synchronize_signals(hr, eda, temp, freq='30S'):

    available_dfs = [df for df in [hr, eda, temp] if not df.empty]

    if not available_dfs:
        return pd.DataFrame()

    start_time = max(df.index.min() for df in available_dfs)
    end_time = min(df.index.max() for df in available_dfs)

    if start_time >= end_time:
        return pd.DataFrame()

    full_index = pd.date_range(start_time, end_time, freq=freq)

    if not hr.empty:
        hr = hr.reindex(
            full_index,
            method='nearest',
            tolerance=pd.Timedelta(seconds=30)
        )
    else:
        hr = pd.DataFrame(index=full_index)

    if not eda.empty:
        eda = eda.reindex(
            full_index,
            method='nearest',
            tolerance=pd.Timedelta(seconds=30)
        )
    else:
        eda = pd.DataFrame(index=full_index)

    if not temp.empty:
        temp = temp.reindex(
            full_index,
            method='nearest',
            tolerance=pd.Timedelta(seconds=300)
        )
    else:
        temp = pd.DataFrame(index=full_index)

    df_final = pd.concat([hr, eda, temp], axis=1)
    df_final.index.name = 'timestamp'

    return df_final

4. Context Feature Engineering

In [ ]:
def add_context_features(df):

    df['hour_of_day'] = (
        df.index.hour + df.index.minute / 60
    )

    df['day_of_week'] = df.index.dayofweek

    df['is_weekend'] = (
        df.index.dayofweek >= 5
    ).astype(int)

    df['is_workhour'] = (
        (df.index.hour >= 9) &
        (df.index.hour < 17)
    ).astype(int)

    return df

5. Stress Label Assignment

In [ ]:
def assign_sparse_stress_labels(df, stress_file, label_ratio=0.1):

    if not os.path.exists(stress_file):
        df['stress_daily'] = np.nan
        return df

    stress_df = pd.read_excel(stress_file)

    stress_df['Date'] = (
        pd.to_datetime(stress_df['Date'])
        .dt.date
    )

    stress_map = dict(
        zip(stress_df['Date'], stress_df['stress'])
    )

    labels = []

    for ts in df.index:
        labels.append(stress_map.get(ts.date(), np.nan))

    df['stress_daily'] = labels

    labeled_idx = []

    for _, group in df.groupby(df.index.date):

        group_idx = group.index.tolist()

        n_label = max(
            1,
            int(len(group_idx) * label_ratio)
        )

        sampled = np.random.choice(
            group_idx,
            size=n_label,
            replace=False
        )

        labeled_idx.extend(sampled)

    df['stress_daily'] = df['stress_daily'].where(
        df.index.isin(labeled_idx),
        np.nan
    )

    return df

6. Signal Cleaning

In [ ]:
def iqr_clip(series):

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return series.clip(lower, upper)


def clean_signals(df):

    numeric_cols = df.select_dtypes(include=[np.number]).columns

    for col in numeric_cols:

        df[col] = (
            df[col]
            .interpolate(method='linear')
            .ffill()
            .bfill()
        )

        df[col] = iqr_clip(df[col])

    return df

7. Feature Scaling and Encoding

In [ ]:
def prepare_ml_dataset(df):

    target_col = 'stress_daily'

    y = df[target_col].values

    df = df.drop(columns=[target_col])

    categorical_cols = (
        df.select_dtypes(include=['object'])
        .columns
        .tolist()
    )

    numeric_cols = (
        df.select_dtypes(include=[np.number])
        .columns
        .tolist()
    )

    scaler = StandardScaler()

    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

    if categorical_cols:

        encoder = OneHotEncoder(
            drop='first',
            sparse_output=False
        )

        encoded = encoder.fit_transform(df[categorical_cols])

        encoded_df = pd.DataFrame(
            encoded,
            columns=encoder.get_feature_names_out(categorical_cols)
        )

        df = pd.concat(
            [df.drop(columns=categorical_cols), encoded_df],
            axis=1
        )

    df[target_col] = y

    return df

8. Main Processing Pipeline

In [ ]:
subjects = sorted([
    d for d in os.listdir(BASE_RAW_PATH)
    if d.startswith('S')
])

for subject_id in subjects:

    print(f"\nProcessing {subject_id}...")

    subject_path = os.path.join(BASE_RAW_PATH, subject_id)

    # Load raw signals
    hr = load_hr(subject_path)
    eda = load_eda(subject_path)
    temp = load_temperature(subject_path)

    # Synchronize signals
    df = synchronize_signals(
        hr,
        eda,
        temp,
        freq=RESAMPLE_FREQ
    )

    if df.empty:
        print(f"Skipping {subject_id}: No valid synchronized data")
        continue

    # Add context features
    df = add_context_features(df)

    # Assign sparse stress labels
    stress_file = os.path.join(
        subject_path,
        'stress',
        f'{subject_id}-stress.xlsx'
    )

    df = assign_sparse_stress_labels(
        df,
        stress_file,
        label_ratio=LABEL_RATIO
    )

    # Clean signals
    df = clean_signals(df)

    # Prepare ML-ready dataset
    df = prepare_ml_dataset(df)

    # Save output
    output_file = os.path.join(
        OUTPUT_PATH,
        f'{subject_id}_processed.csv'
    )

    df.reset_index().to_csv(output_file, index=False)

    print(f"Saved: {output_file}")

9. Final Output

Final processed CSV contains:

1. synchronized wearable signals
2. cleaned physiological features
3. contextual features
4. sparse stress labels
5. normalized ML-ready features


**WESAD Stress Dataset Preprocessing Pipeline**

Configuration

In [ ]:
# =====================================
# CONFIGURATION
# =====================================

INPUT_DIR = "./data/wesad_raw"
OUTPUT_DIR = "./data/wesad_processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

RESAMPLE_WINDOW = "5S"

PANAS Extraction

In [ ]:
def extract_panas_scores(quest_path):

    lines = open(quest_path).readlines()

    panas_lines = [
        line for line in lines
        if line.startswith("# PANAS")
    ]

    scores = []

    for line in panas_lines:

        tokens = line.strip().split(";")[1:25]

        row = [
            int(float(t)) if t.lower() != "nan" else np.nan
            for t in tokens
        ]

        scores.append(row)

    panas = np.array(scores)

    pa_idx = [1, 2, 5, 9, 10, 12, 14, 16, 17, 21]
    na_idx = [0, 3, 6, 7, 11, 13, 15, 18, 19, 20]

    panas_pa = np.nanmean(panas[:, pa_idx], axis=1)
    panas_na = np.nanmean(panas[:, na_idx], axis=1)

    task_names = [
        "Base",
        "TSST",
        "Medi1",
        "Fun",
        "Medi2"
    ][:len(scores)]

    return pd.DataFrame({
        "Task": task_names,
        "PANAS_PA": panas_pa,
        "PANAS_NA": panas_na
    })

Session Timing Extraction

In [ ]:
def extract_session_timings(quest_path):

    lines = open(quest_path).readlines()

    start_line = [
        line for line in lines
        if line.startswith("# START")
    ][0]

    end_line = [
        line for line in lines
        if line.startswith("# END")
    ][0]

    start_times = list(
        map(float, start_line.strip().split(";")[1:6])
    )

    end_times = list(
        map(float, end_line.strip().split(";")[1:6])
    )

    return start_times, end_times

Wrist Signal Resampling

In [ ]:
def resample_wrist_signal(data, signal_name, sampling_rate):

    df = pd.DataFrame({
        signal_name: data["signal"]["wrist"][signal_name].flatten()
    })

    df["Time"] = pd.to_datetime(
        np.arange(len(df)) / sampling_rate,
        unit="s",
        origin="unix"
    )

    df.set_index("Time", inplace=True)

    return df.resample(RESAMPLE_WINDOW).agg([
        "mean",
        "std",
        "min",
        "max",
        "median",
        skew
    ])

Subject Processing Pipeline

In [ ]:
def process_subject(subject_folder):

    subject_id = os.path.basename(subject_folder)

    pkl_path = os.path.join(
        subject_folder,
        f"{subject_id}.pkl"
    )

    quest_xls = os.path.join(
        subject_folder,
        f"{subject_id}_quest.xls"
    )

    quest_csv = os.path.join(
        subject_folder,
        f"{subject_id}_quest.csv"
    )

    if os.path.exists(quest_xls):
        quest_path = quest_xls
    else:
        quest_path = quest_csv

    # =====================================
    # LOAD PICKLE
    # =====================================

    with open(pkl_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    # =====================================
    # CHEST SIGNAL PROCESSING
    # =====================================

    df_chest = pd.DataFrame({

        "ACC_X": data["signal"]["chest"]["ACC"][:, 0],
        "ACC_Y": data["signal"]["chest"]["ACC"][:, 1],
        "ACC_Z": data["signal"]["chest"]["ACC"][:, 2],

        "ECG": data["signal"]["chest"]["ECG"].flatten(),
        "EDA": data["signal"]["chest"]["EDA"].flatten(),
        "Temp": data["signal"]["chest"]["Temp"].flatten(),
        "Resp": data["signal"]["chest"]["Resp"].flatten()
    })

    df_chest["ACC_Mag_Chest"] = np.sqrt(
        df_chest["ACC_X"]**2 +
        df_chest["ACC_Y"]**2 +
        df_chest["ACC_Z"]**2
    )

    df_chest.drop(
        columns=["ACC_X", "ACC_Y", "ACC_Z"],
        inplace=True
    )

    df_chest["Time"] = pd.to_datetime(
        np.arange(len(df_chest)) / 700,
        unit="s",
        origin="unix"
    )

    df_chest.set_index("Time", inplace=True)

    df_chest = df_chest.resample(RESAMPLE_WINDOW).agg([
        "mean",
        "std",
        "min",
        "max",
        "median",
        skew
    ])

    # =====================================
    # WRIST SIGNAL PROCESSING
    # =====================================

    wrist_acc = data["signal"]["wrist"]["ACC"]

    df_acc = pd.DataFrame({
        "ACC_Mag_Wrist": np.sqrt(
            wrist_acc[:, 0]**2 +
            wrist_acc[:, 1]**2 +
            wrist_acc[:, 2]**2
        )
    })

    df_acc["Time"] = pd.to_datetime(
        np.arange(len(df_acc)) / 32,
        unit="s",
        origin="unix"
    )

    df_acc.set_index("Time", inplace=True)

    df_acc = df_acc.resample(RESAMPLE_WINDOW).agg([
        "mean",
        "std",
        "min",
        "max",
        "median",
        skew
    ])

    df_bvp = resample_wrist_signal(data, "BVP", 64)
    df_eda = resample_wrist_signal(data, "EDA", 4)
    df_temp = resample_wrist_signal(data, "TEMP", 4)

    df_wrist = df_acc.join([
        df_bvp,
        df_eda,
        df_temp
    ])

    df_wrist = df_wrist.ffill()

    # =====================================
    # SENSOR FUSION
    # =====================================

    df_final = pd.merge(
        df_chest,
        df_wrist,
        left_index=True,
        right_index=True,
        how="outer"
    )

    df_final = df_final.dropna(axis=1)

    df_final.columns = [
        "_".join(filter(None, col)).strip()
        for col in df_final.columns.values
    ]

    df_final = df_final.reset_index(drop=True)

    # =====================================
    # ACTIVITY CONTEXT
    # =====================================

    mean_acc = df_final["ACC_Mag_Chest_mean"].mean()
    std_acc = df_final["ACC_Mag_Chest_mean"].std()

    low = mean_acc + 0.5 * std_acc
    high = mean_acc + std_acc

    df_final["Activity"] = np.select(
        [
            df_final["ACC_Mag_Chest_mean"] <= low,
            df_final["ACC_Mag_Chest_mean"] > high
        ],
        [
            "Still",
            "HeavyMovement"
        ],
        default="SlightMovement"
    )

    df_final["Posture"] = np.select(
        [
            df_final["ACC_Mag_Chest_mean"] <= 0.85,
            df_final["ACC_Mag_Chest_mean"] > 1.05
        ],
        [
            "Sit",
            "Walk"
        ],
        default="Stand"
    )

    # =====================================
    # TEMPORAL CONTEXT
    # =====================================

    total_windows = len(df_final)

    time_fraction = np.linspace(
        0,
        24,
        total_windows
    )

    df_final["Time_Sin"] = np.sin(
        2 * np.pi * time_fraction / 24
    )

    df_final["Time_Cos"] = np.cos(
        2 * np.pi * time_fraction / 24
    )

    # =====================================
    # TASK CONTEXT
    # =====================================

    session_labels = [
        "Base",
        "TSST",
        "Medi1",
        "Fun",
        "Medi2"
    ]

    starts, ends = extract_session_timings(quest_path)

    starts_idx = [int(x * 12) for x in starts]
    ends_idx = [int(x * 12) for x in ends]

    df_final["Task"] = "Unknown"
    df_final["SessionPhase"] = "Unknown"

    for label, s, e in zip(
        session_labels,
        starts_idx,
        ends_idx
    ):

        df_final.loc[s:e, "Task"] = label

        phase_len = e - s

        df_final.loc[
            s:s + int(0.3 * phase_len),
            "SessionPhase"
        ] = "Initial"

        df_final.loc[
            s + int(0.3 * phase_len):
            s + int(0.7 * phase_len),
            "SessionPhase"
        ] = "Mid"

        df_final.loc[
            s + int(0.7 * phase_len):e,
            "SessionPhase"
        ] = "End"

    df_final = df_final[
        df_final["Task"] != "Unknown"
    ].copy()

    # =====================================
    # PANAS STRESS LABELS
    # =====================================

    scores_df = extract_panas_scores(
        quest_path
    )

    df_final["PANAS_PA"] = np.nan
    df_final["PANAS_NA"] = np.nan

    for _, row in scores_df.iterrows():

        task = row["Task"]

        task_indices = df_final.index[
            df_final["Task"] == task
        ]

        if len(task_indices) == 0:
            continue

        idx = task_indices[-1]

        df_final.at[idx, "PANAS_PA"] = row["PANAS_PA"]
        df_final.at[idx, "PANAS_NA"] = row["PANAS_NA"]

    df_final["stress_raw"] = (
        df_final["PANAS_NA"] -
        df_final["PANAS_PA"]
    ) / 2

    df_final["Stress_Class"] = pd.qcut(
        df_final["stress_raw"].dropna(),
        q=3,
        labels=[
            "LowStress",
            "MediumStress",
            "HighStress"
        ]
    )

    df_final["Stress_Class"] = (
        df_final["Stress_Class"]
        .reindex(df_final.index)
        .ffill(limit=5)
        .bfill(limit=5)
    )

    df_final.drop(
        columns=[
            "PANAS_PA",
            "PANAS_NA",
            "stress_raw"
        ],
        inplace=True,
        errors="ignore"
    )

    return df_final

Encode Categorical Features

In [ ]:
def encode_categorical_columns(df):

    categorical_cols = [
        "Task",
        "SessionPhase",
        "Activity",
        "Posture",
        "Stress_Class"
    ]

    for col in categorical_cols:

        if col in df.columns:

            encoder = LabelEncoder()

            df[col] = encoder.fit_transform(
                df[col].astype(str)
            )

    return df

Main Execution

In [ ]:
subjects = sorted(os.listdir(INPUT_DIR))

for subject in subjects:

    subject_path = os.path.join(
        INPUT_DIR,
        subject
    )

    if not (
        os.path.isdir(subject_path) and
        subject.startswith("S")
    ):
        continue

    print(f"Processing {subject}...")

    try:

        df = process_subject(subject_path)

        df = encode_categorical_columns(df)

        output_path = os.path.join(
            OUTPUT_DIR,
            f"{subject}.csv"
        )

        df.to_csv(output_path, index=False)

        print(f"Saved: {output_path}")

    except Exception as e:

        print(f"Failed {subject}: {e}")

ECG_mean
ECG_std
EDA_mean
Resp_max
ACC_Mag_Chest_skew
BVP_median
Activity
Posture
Task
SessionPhase
Time_Sin
Time_Cos
Stress_Class